# SeizeLUT on the Kria KV260

LUT-native seizure classifier in the FPGA fabric. Run the cells in order (Run → Run All Cells).
`seizelut.bit` and `seizelut.hwh` must be in this folder. The full test campaign is the last cell.

In [ ]:
import sys, json, numpy as np
sys.path.insert(0, '.')
from seizelut_kv260 import SeizeLUT
from seizelut_model import SeizeLUTModel
core = SeizeLUT('pynq', bitfile='seizelut.bit')
print('core at', hex(core.base))
core.identify()

## Built-in self-test
1,024 golden vectors stored in on-chip memory run through the classifier at one window per clock; the hardware compares every score, decision and alarm.

In [ ]:
core.bist()

## One recording through the FPGA
Every one-second window is written to the core over AXI4-Lite; the results are compared with the software model.

In [ ]:
rec = 'sub-098_run-03'
z = np.load(f'vectors/{rec}.npz')
core.set_operating_point(2, 10)   # programmable path: the <= 1 FA/h point
hw = core.run_recording(z['feats'], z['art'])
sw = SeizeLUTModel('model').run(z['feats'], z['art'])
print(rec, len(z['feats']), 'windows in', round(hw['seconds'], 2), 's via', hw['method'])
print('mismatches vs software:', int((hw['score'] != sw['score']).sum() + (hw['alarm'] != sw['alarm']).sum()))

In [ ]:
import matplotlib.pyplot as plt
on, off = z['seizures'][0]
t = np.arange(int(on) - 300, int(off) + 300)
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
ax[0].plot(t, hw['score'][t]); ax[0].axhline(9, ls='--', c='gray'); ax[0].set_ylabel('FPGA score')
ax[1].step(t, hw['alarm'][t], where='post', label='frozen point'); ax[1].step(t, hw['alarm2'][t] * 0.8, where='post', label='<= 1 FA/h point')
ax[1].legend(); ax[1].set_ylabel('alarm')
ax[2].step(t, z['y'][t], where='post', c='k'); ax[2].set_ylabel('seizure'); ax[2].set_xlabel('seconds')
for a in ax: a.axvspan(on, off, color='orange', alpha=0.2)
plt.tight_layout(); plt.show()

## Full test campaign
All 10 recordings (431,960 windows), detection metrics and power. Takes a few minutes; results are saved in `results/`.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, 'run_tests.py'], capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-2000:])